# Task 6: Partition Strategy Benchmark

This notebook compares two different Delta table storage designs for the
**taxi_trips** dataset (the raw fact table from Task 3).

**What is held constant (so the comparison is fair):**
- Same source rows, same schema, same Spark session, same machine.
- Both tables are written back-to-back in this single run, so JVM warm-up
  and OS-level effects apply roughly equally to both.

**What is different between the two designs:**
- **Strategy A - time-based:** partitioned by `pickup_year`, `pickup_month`
- **Strategy B - location-based:** partitioned by `pickup_borough`.

We measure, for each strategy:
1. Ingestion time (how long the `.write` takes)
2. Storage size on disk
3. Number of output files
4. Query latency for the three required queries:
   - number of trips per borough
   - average trip duration per day
   - average fare per borough

## 1. Setup

In [1]:
import os
import shutil
import time
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

In [2]:
builder = (
    SparkSession.builder
    .appName("Task6-Partition-Benchmark")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1cccfcb5-6bcc-42de-9c65-ec5683429fdb;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Spark version: 4.2.0


In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a parent directory containing data/raw")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

BENCHMARK_DIR = PROCESSED_DIR / "benchmark"
STRATEGY_A_PATH = BENCHMARK_DIR / "taxi_trips_by_time"
STRATEGY_B_PATH = BENCHMARK_DIR / "taxi_trips_by_borough"

TAXI_TRIPS_PATH = PROCESSED_DIR / "taxi_trips"
TAXI_ZONES_PATH = PROCESSED_DIR / "taxi_zones"

for path in (TAXI_TRIPS_PATH, TAXI_ZONES_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing processed Delta table: {path}. Run ingestion_framework.ipynb first."
        )

print("Taxi trips source:", TAXI_TRIPS_PATH)
print("Strategy A (time) output:", STRATEGY_A_PATH)
print("Strategy B (borough) output:", STRATEGY_B_PATH)

Taxi trips source: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Strategy A (time) output: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/benchmark/taxi_trips_by_time
Strategy B (borough) output: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/benchmark/taxi_trips_by_borough


## 2. Build one shared source DataFrame

Both strategies must write the *exact same data*, so the only variable being
tested is the partitioning scheme. We add `pickup_borough` once here (by
joining the small `taxi_zones` lookup table) and reuse this single DataFrame
for both writes.

In [4]:
trips = spark.read.format("delta").load(str(TAXI_TRIPS_PATH))
zones = spark.read.format("delta").load(str(TAXI_ZONES_PATH))

pickup_zones = F.broadcast(
    zones.select(
        F.col("location_id").alias("pickup_zone_location_id"),
        F.col("borough").alias("pickup_borough"),
    )
)

benchmark_df = (
    trips
    .join(pickup_zones, trips.pickup_location_id == pickup_zones.pickup_zone_location_id, "left")
    .drop("pickup_zone_location_id")
    .withColumn("pickup_borough", F.coalesce(F.col("pickup_borough"), F.lit("Unknown")))
)
benchmark_df = benchmark_df.cache()
row_count = benchmark_df.count()
print("Rows to write for each strategy:", f"{row_count:,}")

zones.filter(F.col("borough") == "N/A").show()
zones.filter(F.col("borough") == "Unknown").show()

26/09/12 15:19:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Rows to write for each strategy: 9,417,689
+-----------+-------+--------------+------------+--------------------+--------------------+--------------------+---------------+
|location_id|borough|          zone|service_zone|        _source_file|   _ingestion_run_id|        _ingested_at|_schema_version|
+-----------+-------+--------------+------------+--------------------+--------------------+--------------------+---------------+
|        265|    N/A|Outside of NYC|         N/A|file:///Users/rex...|7aeb01a6-19f8-41b...|2026-09-10 05:36:...|          1.0.0|
+-----------+-------+--------------+------------+--------------------+--------------------+--------------------+---------------+

+-----------+-------+----+------------+--------------------+--------------------+--------------------+---------------+
|location_id|borough|zone|service_zone|        _source_file|   _ingestion_run_id|        _ingested_at|_schema_version|
+-----------+-------+----+------------+--------------------+-------------

## 3. Helper functions

- `time_write(...)`: writes the DataFrame with a given partitioning and
  returns how long it took.
- `describe_storage(...)`: walks the output folder and reports total size
  and file count.
- `time_query(...)`: runs a query and returns both the result and how long
  it took to fully execute.

In [5]:
def time_write(df, path: Path, partition_cols: list[str]) -> float:
    if path.exists():
        shutil.rmtree(path)  # start clean so old runs never leak into the size/file-count numbers
    started = time.perf_counter()
    (
        df
        # One file per partition value, instead of letting each of the 64
        .repartition(*[F.col(c) for c in partition_cols])
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy(*partition_cols)
        .save(str(path))
    )
    return time.perf_counter() - started


def describe_storage(path: Path) -> dict:
    total_bytes = 0
    file_count = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [d for d in dirs if d != "_delta_log"]
        for name in files:
            total_bytes += (Path(root) / name).stat().st_size
            file_count += 1
    return {"size_mb": total_bytes / (1024 ** 2), "file_count": file_count}


def time_query(df) -> tuple[list, float]:
    started = time.perf_counter()
    rows = df.collect()
    elapsed = time.perf_counter() - started
    return rows, elapsed

## 4. Write both strategies and measure ingestion + storage

In [6]:
results = {}

ingestion_time_a = time_write(benchmark_df, STRATEGY_A_PATH, ["pickup_year", "pickup_month"])
results["strategy_a_time_based"] = {
    "partition_columns": ["pickup_year", "pickup_month"],
    "ingestion_time_seconds": ingestion_time_a,
    **describe_storage(STRATEGY_A_PATH),
}

ingestion_time_b = time_write(benchmark_df, STRATEGY_B_PATH, ["pickup_borough"])
results["strategy_b_borough_based"] = {
    "partition_columns": ["pickup_borough"],
    "ingestion_time_seconds": ingestion_time_b,
    **describe_storage(STRATEGY_B_PATH),
}

# We no longer need the cached source now that both copies are written.
benchmark_df.unpersist()

for name, stats in results.items():
    print(name, stats)

strategy_a_time_based {'partition_columns': ['pickup_year', 'pickup_month'], 'ingestion_time_seconds': 12.235706832958385, 'size_mb': 842.7888679504395, 'file_count': 16}
strategy_b_borough_based {'partition_columns': ['pickup_borough'], 'ingestion_time_seconds': 22.881501292111352, 'size_mb': 845.7840309143066, 'file_count': 16}


## 5. Run the three required queries against each strategy

Each strategy is **re-read from disk** here rather than reused from the
cached `benchmark_df` above, so timings reflect a real, cold query against
each *stored table*, not an in-memory cache.

To avoid one strategy systematically benefiting from JVM warm-up just
because it happens to run first, this section:
1. Runs a discard-the-result warm-up query against **both** strategies
   before any timed measurement, so both start from an equally warm JVM.
2. Times each (strategy, query) pair **5 times**, alternating which
   strategy goes first each round, and reports the **median** — a single
   run is too noisy to draw conclusions from.

In [7]:
strategy_a = spark.read.format("delta").load(str(STRATEGY_A_PATH))
strategy_b = spark.read.format("delta").load(str(STRATEGY_B_PATH))

import statistics

queries = {
    "trips_per_borough": lambda df: df.groupBy("pickup_borough").count(),
    "avg_trip_duration_per_day": lambda df: df.groupBy("pickup_date").agg(
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes")
    ),
    "avg_fare_per_borough": lambda df: df.groupBy("pickup_borough").agg(
        F.avg("fare_amount").alias("avg_fare_amount")
    ),
}

strategies = [
    ("strategy_a_time_based", strategy_a),
    ("strategy_b_borough_based", strategy_b),
]

# --- Warm-up (not measured): forces JVM codegen for both strategies before
# either one gets a timing advantage from simply going first. ---
for _, df in strategies:
    for query_fn in queries.values():
        query_fn(df).collect()

# --- Timed phase: 5 repetitions, alternating which strategy goes first
# each round, taking the median per (strategy, query) pair. ---
N_REPEATS = 5
raw_timings: dict[tuple[str, str], list[float]] = {}

for rep in range(N_REPEATS):
    ordered = strategies if rep % 2 == 0 else list(reversed(strategies))
    for strategy_name, df in ordered:
        for query_name, query_fn in queries.items():
            _, elapsed = time_query(query_fn(df))
            raw_timings.setdefault((strategy_name, query_name), []).append(elapsed)

query_results = {
    key: statistics.median(vals) for key, vals in raw_timings.items()
}
for (strategy_name, query_name), median_t in sorted(query_results.items()):
    all_runs = ", ".join(f"{v:.2f}" for v in raw_timings[(strategy_name, query_name)])
    print(f"{strategy_name:25s} {query_name:28s} median={median_t:.3f}s  (runs: {all_runs})")

strategy_a_time_based     avg_fare_per_borough         median=0.148s  (runs: 0.16, 0.14, 0.15, 0.15, 0.15)
strategy_a_time_based     avg_trip_duration_per_day    median=0.142s  (runs: 0.14, 0.14, 0.14, 0.14, 0.14)
strategy_a_time_based     trips_per_borough            median=0.139s  (runs: 0.18, 0.14, 0.14, 0.13, 0.14)
strategy_b_borough_based  avg_fare_per_borough         median=0.133s  (runs: 0.16, 0.11, 0.13, 0.13, 0.14)
strategy_b_borough_based  avg_trip_duration_per_day    median=0.147s  (runs: 0.18, 0.15, 0.15, 0.14, 0.15)
strategy_b_borough_based  trips_per_borough            median=0.119s  (runs: 0.12, 0.12, 0.11, 0.12, 0.12)


## 6. Combine everything into one comparison table

In [8]:
import pandas as pd

rows = []
for strategy_name, stats in results.items():
    row = {
        "strategy": strategy_name,
        "partition_columns": ", ".join(stats["partition_columns"]),
        "ingestion_time_s": round(stats["ingestion_time_seconds"], 2),
        "storage_size_mb": round(stats["size_mb"], 1),
        "file_count": stats["file_count"],
    }
    for query_name in queries:
        row[f"{query_name}_s"] = round(query_results[(strategy_name, query_name)], 3)
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df

,strategy,partition_columns,ingestion_time_s,storage_size_mb,file_count,trips_per_borough_s,avg_trip_duration_per_day_s,avg_fare_per_borough_s
0,strategy_a_time_based,"pickup_year, pickup_month",12.24,842.8,16,0.139,0.142,0.148
1,strategy_b_borough_based,pickup_borough,22.88,845.8,16,0.119,0.147,0.133


## 7. Row-count distribution by partition key

This section computes the row-count breakdown behind Tables 1 and 2 in the
report (Section 3.1, Data distribution and quality), and the largest/smallest
group sizes used in Section 3.2 to explain the ingestion-time gap between
strategies.

A handful of rows have malformed `pickup_year` values (2002, 2008, 2009),
which would make the time-based grouping look artificially skewed if
included at face value (a group with only 1-2 rows). The breakdown is
therefore computed twice: once on the raw data, for transparency, and once
restricted to `pickup_year = 2024`, which is the comparison used in the
report.

In [9]:
def skew_ratio(counts_df) -> tuple[int, int, float]:
    row = counts_df.agg(F.max("count").alias("mx"), F.min("count").alias("mn")).collect()[0]
    return row["mx"], row["mn"], row["mx"] / row["mn"]

# --- Raw data, all years (includes the known malformed-timestamp rows) ---
borough_counts_raw = strategy_b.groupBy("pickup_borough").count().orderBy(F.desc("count"))
time_counts_raw = strategy_a.groupBy("pickup_year", "pickup_month").count().orderBy(F.desc("count"))

print("Borough group sizes (raw):")
borough_counts_raw.show()
print("Year/month group sizes (raw):")
time_counts_raw.show(20)

b_max, b_min, b_ratio = skew_ratio(borough_counts_raw)
t_max, t_min, t_ratio = skew_ratio(time_counts_raw)
print(f"Borough skew (raw):     largest={b_max:,} smallest={b_min:,} ratio={b_ratio:,.1f}x")
print(f"Year/month skew (raw):  largest={t_max:,} smallest={t_min:,} ratio={t_ratio:,.1f}x")

# --- Fair comparison: 2024 only, excludes the malformed-year rows on both sides ---
clean_a = strategy_a.filter(F.col("pickup_year") == 2024)
clean_b = strategy_b.filter(F.col("pickup_year") == 2024)

month_counts = clean_a.groupBy("pickup_month").count().orderBy(F.desc("count"))
borough_counts_clean = clean_b.groupBy("pickup_borough").count().orderBy(F.desc("count"))

print("\nMonth group sizes (2024 only):")
month_counts.show()
print("Borough group sizes (2024 only):")
borough_counts_clean.show()

m_max, m_min, m_ratio = skew_ratio(month_counts)
c_max, c_min, c_ratio = skew_ratio(borough_counts_clean)
print(f"Month skew (2024 only):   largest={m_max:,} smallest={m_min:,} ratio={m_ratio:,.1f}x")
print(f"Borough skew (2024 only): largest={c_max:,} smallest={c_min:,} ratio={c_ratio:,.1f}x")

Borough group sizes (raw):
+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|8442887|
|        Queens| 817093|
|      Brooklyn|  95770|
|       Unknown|  31304|
|         Bronx|  24727|
|           N/A|   4773|
|           EWR|    915|
| Staten Island|    220|
+--------------+-------+

Year/month group sizes (raw):
+-----------+------------+-------+
|pickup_year|pickup_month|  count|
+-----------+------------+-------+
|       2024|           3|3523884|
|       2024|           2|2966792|
|       2024|           1|2926993|
|       2023|          12|     10|
|       2009|           1|      4|
|       2002|          12|      3|
|       2024|           4|      2|
|       2008|          12|      1|
+-----------+------------+-------+

Borough skew (raw):     largest=8,442,887 smallest=220 ratio=38,376.8x
Year/month skew (raw):  largest=3,523,884 smallest=1 ratio=3,523,884.0x

Month group sizes (2024 only):
+------------+-------+
|pickup_month|  count|


## 8. Verifying partition pruning

This section produces the evidence behind Section 3.4 of the report
(Partition pruning): confirmation that Spark's pruning mechanism triggers
specifically on the partition column, and the majority-vs-minority filter
timings reported in Table 5.

Two checks are performed:

1. **Physical plan inspection (`.explain()`)** — a filter on the partition
   column should produce a non-empty `PartitionFilters` list; a filter on
   any other column should produce an empty one, with the condition instead
   appearing as a `DataFilter`.
2. **Timing** — for each strategy, a full scan is compared against a filter
   on its *majority* partition value and a filter on its *minority* value,
   using `sum(fare_amount)` to force Spark to actually read the column
   rather than answer from Delta's row-count metadata. As in Section 5,
   both strategies are warmed up first, the three variants per strategy are
   timed over 5 repetitions with the order rotated each time, and the
   median is reported.

In [10]:
print("=== Strategy A: filter on the PARTITION column (pickup_year) ===")
strategy_a.filter(F.col("pickup_year") == 2024).select("trip_id").explain()

print("\n=== Strategy A: filter on a NON-partition column (passenger_count) ===")
strategy_a.filter(F.col("passenger_count") == 1).select("trip_id").explain()

print("\n=== Strategy B: filter on the PARTITION column (pickup_borough) ===")
strategy_b.filter(F.col("pickup_borough") == "Manhattan").select("trip_id").explain()

print("\n=== Strategy B: filter on a NON-partition column (passenger_count) ===")
strategy_b.filter(F.col("passenger_count") == 1).select("trip_id").explain()

=== Strategy A: filter on the PARTITION column (pickup_year) ===
== Physical Plan ==
*(1) Project [trip_id#6005]
+- *(1) ColumnarToRow
   +- FileScan parquet [trip_id#6005,pickup_year#6009,pickup_month#6010] Batched: true, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intens..., PartitionFilters: [isnotnull(pickup_year#6009), (pickup_year#6009 = 2024)], PushedFilters: [], ReadSchema: struct<trip_id:string>



=== Strategy A: filter on a NON-partition column (passenger_count) ===
== Physical Plan ==
*(1) Project [trip_id#6005]
+- *(1) Filter (isnotnull(passenger_count#5985) AND (passenger_count#5985 = 1))
   +- *(1) ColumnarToRow
      +- FileScan parquet [passenger_count#5985,trip_id#6005,pickup_year#6009,pickup_month#6010] Batched: true, DataFilters: [isnotnull(passenger_count#5985), (passenger_count#5985 = 1)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[file:/Users/rexchang/Docume

In [11]:
# def time_agg(df, col) -> tuple[float, float]:
#     started = time.perf_counter()
#     total = df.agg(F.sum(col).alias("total")).collect()[0]["total"]
#     return total, time.perf_counter() - started

# # Run the minority-category filters
# total_full_a, t_full_a = time_agg(strategy_a, "fare_amount")
# total_april, t_april = time_agg(
#     strategy_a.filter((F.col("pickup_year") == 2024) & (F.col("pickup_month") == 4)),
#     "fare_amount",
# )
# print(f"Strategy A full scan (sum fare_amount):        {t_full_a:.2f}s")
# print(f"Strategy A April 2024 only (sum fare_amount):  {t_april:.2f}s")
# print(f"Speedup: {t_full_a / t_april:.1f}x\n")

# total_full_b, t_full_b = time_agg(strategy_b, "fare_amount")
# total_staten, t_staten = time_agg(
#     strategy_b.filter(F.col("pickup_borough") == "Staten Island"), "fare_amount"
# )
# print(f"Strategy B full scan (sum fare_amount):            {t_full_b:.2f}s")
# print(f"Strategy B Staten Island only (sum fare_amount):   {t_staten:.2f}s")
# print(f"Speedup: {t_full_b / t_staten:.1f}x")

In [12]:
# # Run the majority-category filters
# total_full_a, t_full_a = time_agg(strategy_a, "fare_amount")
# total_2024, t_2024 = time_agg(strategy_a.filter(F.col("pickup_year") == 2024), "fare_amount")
# print(f"Strategy A full scan (sum fare_amount):         {t_full_a:.2f}s")
# print(f"Strategy A pickup_year=2024 (sum fare_amount):  {t_2024:.2f}s")
# print(f"Speedup: {t_full_a / t_2024:.1f}x\n")

# total_full_b, t_full_b = time_agg(strategy_b, "fare_amount")
# total_manhattan, t_manhattan = time_agg(
#     strategy_b.filter(F.col("pickup_borough") == "Manhattan"), "fare_amount"
# )
# print(f"Strategy B full scan (sum fare_amount):              {t_full_b:.2f}s")
# print(f"Strategy B pickup_borough='Manhattan' (sum fare_amount): {t_manhattan:.2f}s")
# print(f"Speedup: {t_full_b / t_manhattan:.1f}x")

In [13]:
import statistics

def time_agg(df, col) -> float:
    started = time.perf_counter()
    df.agg(F.sum(col).alias("total")).collect()
    return time.perf_counter() - started

strategy_dfs = {
    "strategy_a_time_based": strategy_a,
    "strategy_b_borough_based": strategy_b,
}

pruning_variants = {
    "strategy_a_time_based": {
        "full_scan": lambda df: df,
        "majority (pickup_year=2024)": lambda df: df.filter(F.col("pickup_year") == 2024),
        "minority (pickup_month=4 in 2024)": lambda df: df.filter(
            (F.col("pickup_year") == 2024) & (F.col("pickup_month") == 4)
        ),
    },
    "strategy_b_borough_based": {
        "full_scan": lambda df: df,
        "majority (pickup_borough='Manhattan')": lambda df: df.filter(F.col("pickup_borough") == "Manhattan"),
        "minority (pickup_borough='Staten Island')": lambda df: df.filter(F.col("pickup_borough") == "Staten Island"),
    },
}

# --- Warm-up (not measured): forces JVM codegen for every variant before
# timing starts, so none of them gets an unfair "went first" advantage. ---
for strategy_name, variants in pruning_variants.items():
    base_df = strategy_dfs[strategy_name]
    for variant_fn in variants.values():
        time_agg(variant_fn(base_df), "fare_amount")

# --- Timed phase: 5 repetitions, rotating which of the 3 variants goes
# first each round, taking the median per (strategy, variant) pair. ---
N_REPEATS = 5
raw_pruning_timings: dict[tuple[str, str], list[float]] = {}

for rep in range(N_REPEATS):
    for strategy_name, variants in pruning_variants.items():
        base_df = strategy_dfs[strategy_name]
        items = list(variants.items())
        offset = rep % len(items)
        rotated = items[offset:] + items[:offset]
        for variant_name, variant_fn in rotated:
            elapsed = time_agg(variant_fn(base_df), "fare_amount")
            raw_pruning_timings.setdefault((strategy_name, variant_name), []).append(elapsed)

pruning_medians = {
    key: statistics.median(vals) for key, vals in raw_pruning_timings.items()
}
for (strategy_name, variant_name), median_t in sorted(pruning_medians.items()):
    all_runs = ", ".join(f"{v:.3f}" for v in raw_pruning_timings[(strategy_name, variant_name)])
    print(f"{strategy_name:25s} {variant_name:40s} median={median_t:.3f}s  (runs: {all_runs})")

print()
for strategy_name, variants in pruning_variants.items():
    full = pruning_medians[(strategy_name, "full_scan")]
    for variant_name in variants:
        if variant_name == "full_scan":
            continue
        filtered = pruning_medians[(strategy_name, variant_name)]
        print(f"{strategy_name:25s} {variant_name:40s} speedup={full / filtered:.2f}x")

strategy_a_time_based     full_scan                                median=0.093s  (runs: 0.123, 0.096, 0.092, 0.085, 0.093)
strategy_a_time_based     majority (pickup_year=2024)              median=0.098s  (runs: 0.100, 0.103, 0.098, 0.087, 0.091)
strategy_a_time_based     minority (pickup_month=4 in 2024)        median=0.085s  (runs: 0.089, 0.095, 0.082, 0.085, 0.080)
strategy_b_borough_based  full_scan                                median=0.091s  (runs: 0.101, 0.095, 0.091, 0.088, 0.089)
strategy_b_borough_based  majority (pickup_borough='Manhattan')    median=0.091s  (runs: 0.100, 0.089, 0.091, 0.089, 0.098)
strategy_b_borough_based  minority (pickup_borough='Staten Island') median=0.084s  (runs: 0.094, 0.082, 0.083, 0.084, 0.086)

strategy_a_time_based     majority (pickup_year=2024)              speedup=0.95x
strategy_a_time_based     minority (pickup_month=4 in 2024)        speedup=1.10x
strategy_b_borough_based  majority (pickup_borough='Manhattan')    speedup=1.00x
strategy_b_

## 9. Note on the timing results above

The speedups printed at the end of Section 8 (0.95x–1.10x) are not a
reliable measurement, despite following the same warm-up and
median-of-5-repetitions methodology used successfully in Section 5. The
full-table scan is itself one of the variants included in that warm-up: it
runs once, discarded, before any timed repetition begins. By the time
timing starts, Spark and the OS file-system cache have already fully read
both strategies' underlying Parquet files, so the full scan and the highly
selective filters end up reading from the same warm cache and take
essentially the same time.

This is a structural limitation of measuring wall-clock I/O savings within
a single, repeated-run session on a dataset this size (≈843 MB total,
comfortably cached by a 48 GB machine): once any query has touched the
files once, every later query benefits equally, and a Jupyter kernel
restart does not clear the OS-level page cache that causes this.

The report (Section 3.4) therefore relies only on the `.explain()` evidence
in Section 8 to confirm that partition pruning triggers correctly on the
partition column, and does not report a wall-clock speedup number.